# Change from V2.0:
## Updated Secondary Label detection and allication in labeling stage

In [2]:
# %% [markdown]
# # RQ2 — Step 0: Clone repositories
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# and writes a manifest with basic metadata.

# %%
from __future__ import annotations
import csv, subprocess, sys, json, time
from pathlib import Path
from typing import Optional, List
from pathlib import Path

# -----------------------------
# Config (edit as needed)
# -----------------------------


# Use a raw string r"..." for Windows paths with spaces
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"   # put your CSV here
CLONE_ROOT   = WORK_ROOT / "clones"         # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifest.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_cloned(url: str, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)
    if d.exists() and (d / ".git").exists():
        # Refresh remote info (best-effort)
        try:
            sh(["git", "fetch", "--all", "--tags", "--prune"], cwd=d)
        except Exception:
            pass
        return d
    sh(["git", "clone", "--no-tags", "--filter=blob:none", "--recurse-submodules=no", url, str(d)])
    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir)
    return int(cp.stdout.strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {"repo_url": url, "dir": None, "status": "unknown", "seconds": None, "total_commits": None, "error": ""}
        try:
            d = ensure_cloned(url, CLONE_ROOT)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["repo_url","dir","status","seconds","total_commits","error"])
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\connectbot__connectbot (1.64s)
[ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\robolectric__robolectric (7.22s)
[ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\opendocument-app__OpenDocument.droid (4.29s)
[ok] https://github.com/maxpower47/PinDroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\maxpower47__PinDroid (2.95s)
[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\Rajawali__Rajawali (6.46s)
[ok] https://github.com/cgeo/cgeo -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\cgeo__cgeo (24.03s)
[ok] https://github.com/OneBusAway/onebusaway-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\OneBusAway__onebus

In [4]:
# %% [markdown]
# RQ2 — Step 1: Mine commit snapshots (simplified invocation feature)
# Scans cloned repos for commits touching CI/YAML/Gradle/scripts and writes per-repo JSONL snapshots.
# - Windows-safe UTF-8 decoding for all git calls (fixes cp1252 UnicodeDecodeError).
# - Robust fallbacks so a wonky repo doesn't stop the whole run.
# - Reduced script false positives (extensions only).
# - Canonical headless tag (no duplicate "no_window").
# - Invocation tags also detected in Gradle DSL.
# - Richer AGP & Orchestrator detection.
# - Removed unused RE_VERSION.

# %%
from __future__ import annotations

import os
import re
import json
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any, Set

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clones"
SNAPSHOT_DIR   = WORK_ROOT / "snapshots"   # per-repo JSONL output here
MAX_COMMITS_PER_REPO = 0                   # 0 = no limit

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# Optional YAML support (safe to skip if you don't need deep YAML parsing)
try:
    import yaml  # pip install pyyaml
except Exception:
    yaml = None

# -----------------------------
# Subprocess helper (Windows-safe UTF-8)
# -----------------------------
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """
    Run a command and return CompletedProcess with UTF-8 decoding and error replacement.
    Prevents UnicodeDecodeError on Windows when reading git output.
    """
    env = os.environ.copy()
    # Disable the pager portably
    env["GIT_PAGER"] = ""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",   # force UTF-8 decode
        errors="replace",   # never crash on odd bytes
        env=env,
    )

# -----------------------------
# Relevant file surfaces
# -----------------------------
CI_FILES = [
    ".gitlab-ci.yml",
    ".circleci/config.yml",
    "bitrise.yml",
    "azure-pipelines.yml",
]

GRADLE_FILES = [
    "build.gradle", "build.gradle.kts",
    "settings.gradle", "settings.gradle.kts",
    "gradle.properties",
    "gradle/wrapper/gradle-wrapper.properties",
]

# Script extensions only (no substring "script" to avoid false positives)
SCRIPT_EXTS = {".sh", ".bash", ".zsh", ".py", ".bat", ".cmd", ".ps1", ".psm1"}

def is_ci_file(p: str) -> bool:
    return p.startswith(".github/workflows/") or p in CI_FILES

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(ext) for ext in SCRIPT_EXTS)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip():
            continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

# -----------------------------
# Git helpers
# -----------------------------
def list_relevant_commits(repo_dir: Path) -> List[Tuple[str, int, List[str]]]:
    """
    Returns list of (sha, unix_ts, [changed_paths]) for commits that touched relevant files.
    Oldest -> newest order.
    """
    cp = sh([
        "git", "-c", "i18n.logOutputEncoding=UTF-8", "-c", "core.quotepath=off",
        "log", "--all", "--name-only", "--pretty=%H%x09%ct"
    ], cwd=repo_dir)
    results: List[Tuple[str, int, List[str]]] = []
    sha: Optional[str] = None
    ts: Optional[int] = None
    changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            if sha is not None and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1)
            ts = int(ts_s)
            changed = []
        else:
            if line.strip():
                changed.append(line.strip())
    if sha is not None and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()  # oldest -> newest
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    try:
        cp = sh(["git", "-c", "i18n.logOutputEncoding=UTF-8", "show", "-s", "--format=%s", sha],
                cwd=repo_dir, check=False)
        return (cp.stdout or "").strip()
    except Exception:
        return ""

# -----------------------------
# Heuristic extractors
# -----------------------------
RE_INT = re.compile(r"\d+")

YAML_KEYS = {
    "api": ["api-level","apilevel","api_level"],
    "abi": ["abi","arch","cpu","abi_filters","abi-filter"],
    "system_image": ["system-image","target","systemimage"],
    "device": ["device","avd-name","avd","device-profile","model","hardwareProfile"],
    "orchestrator": ["orchestrator","android-test-orchestrator","use-orchestrator"],
    "wait": ["wait-for-boot","wait_for_boot"],
    "timeouts": ["emulator-boot-timeout","timeout","test-timeout","emulator_timeout"],
    "retries": ["retry","retries","max-retries"],
    "matrix": ["matrix","strategy"],
    "runner_os": ["runs-on","machine","image"],
    "jdk": ["java-version","jdk","java","distribution"],
    "invocation": ["run","gradle_args","gradlew_args","task","tasks"],
    "thirdparty": ["browserstack","saucelabs","firebase","bitbar","kobiton","testlab","devicefarm"],
}

# --- Simplified invocation classification (2D: style + tags) ---

DIY_RX = re.compile(
    r"(?:\bemulator(?:\.bat)?\s-|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b|"
    r"\bcreate\s+avd\b|"
    r"\badb\s+(?:-s\s+\S+\s+)?wait-for-device\b|"
    r"\badb\s+shell\s+getprop\s+sys\.boot_completed\b|"
    r"\bqemu\b)",
    re.I,
)

GMD_RX = re.compile(
    r"(?:\bmanaged\s+device\b|\bgradle\s+managed\s+device\b|\bPixel\w*Api\d+\b)",
    re.I,
)

GMD_GRADLE_RX = re.compile(
    r"(?:\bmanagedDevices\s*\{|testOptions\s*\{[^}]*devices)",
    re.I | re.S,
)

CONNECTED_RX = re.compile(r"\bconnectedAndroidTest\b", re.I)

# Canonical tag set from YAML CLI snippets
TAG_PATTERNS_YAML = {
    "headless":    re.compile(r"-no-window|-headless", re.I),   # canonical
    "gpu":         re.compile(r"-gpu\s+\w+", re.I),
    "concurrency": re.compile(r"--parallel|\bmax-workers\b|\borg\.gradle\.workers\.max\b", re.I),
    "sharding":    re.compile(r"\bnumShards\b|\bshard(?:ing|Index)?\b", re.I),
    "instr_args":  re.compile(r"-Pandroid\.testInstrumentationRunnerArguments\.", re.I),
}

# Tags detectable in Gradle DSL / properties
TAG_PATTERNS_GRADLE = {
    "sharding":    re.compile(r"testInstrumentationRunnerArguments(?:\[[\"']|\.)(?:numShards|shardIndex)", re.I),
    "instr_args":  re.compile(r"testInstrumentationRunnerArguments", re.I),
    "concurrency": re.compile(r"\borg\.gradle\.workers\.max\b|\bmaxWorkers\b", re.I),
}

def _collect_tags_from_text(text: str, patterns: Dict[str, re.Pattern]) -> Set[str]:
    return {name for name, rx in patterns.items() if rx.search(text or "")}

def classify_invocation_from_texts(yaml_snippets: List[str], gradle_texts: List[str]) -> tuple[str, List[str]]:
    """
    Returns (invocation_style, invocation_tags)
    - yaml_snippets: values from 'run'/'task'/'args' collected from YAML
    - gradle_texts: full text of any Gradle files in the same snapshot (optional)
    """
    hay_yaml = "\n".join(s for s in yaml_snippets if s)
    hay_gradle = "\n".join(gradle_texts or [])

    gmd_hit = bool(GMD_RX.search(hay_yaml)) or any(GMD_GRADLE_RX.search(t or "") for t in gradle_texts or [])
    diy_hit = bool(DIY_RX.search(hay_yaml) or DIY_RX.search(hay_gradle))
    connected_hit = bool(CONNECTED_RX.search(hay_yaml) or CONNECTED_RX.search(hay_gradle))

    if gmd_hit:
        style = "gmd"
    elif diy_hit:
        style = "diy"
    elif connected_hit:
        style = "gradle_connected"
    else:
        style = "unknown"

    tags_yaml = _collect_tags_from_text(hay_yaml, TAG_PATTERNS_YAML)
    tags_gradle = set()
    for t in gradle_texts or []:
        tags_gradle |= _collect_tags_from_text(t, TAG_PATTERNS_GRADLE)

    tags = sorted(tags_yaml | tags_gradle)
    return style, tags

# -----------------------------
# YAML extractor
# -----------------------------
def extract_from_yaml_text(text: str, gradle_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    if yaml is None:
        return {}
    try:
        docs = list(yaml.safe_load_all(text))
    except Exception:
        docs = []
    out = {
        # real features
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        "orchestrator": None, "wait_for_boot": None, "timeouts": {}, "retries": None,
        "matrix_axes": set(), "runner_os": None, "jdk": None,
        # study-defined (simplified)
        "invocation_style": "unknown", "invocation_tags": set(),
        "thirdparty_refs": set(),
    }
    # temp bucket: raw run/task/args strings for classification
    _invocation_snippets: List[str] = []

    def scan_obj(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                lk = str(k).lower()
                if lk in (x.lower() for x in YAML_KEYS["api"]):
                    if isinstance(v, list):
                        for x in v:
                            if isinstance(x, (int,str)) and RE_INT.search(str(x)):
                                out["api_levels"].add(int(RE_INT.search(str(x)).group()))
                    elif isinstance(v, (int,str)):
                        m = RE_INT.search(str(v))
                        if m: out["api_levels"].add(int(m.group()))
                if lk in (x.lower() for x in YAML_KEYS["abi"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["abis"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["system_image"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["system_images"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["device"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["device_profiles"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["orchestrator"]):
                    if isinstance(v, bool): out["orchestrator"] = v
                    elif isinstance(v, str): out["orchestrator"] = v.lower() in ("1","true","yes","on")
                if lk in (x.lower() for x in YAML_KEYS["wait"]):
                    if isinstance(v, bool): out["wait_for_boot"] = v
                    elif isinstance(v, str): out["wait_for_boot"] = v.lower() in ("1","true","yes","on")
                if lk in (x.lower() for x in YAML_KEYS["timeouts"]):
                    out["timeouts"][k] = v
                if lk in (x.lower() for x in YAML_KEYS["retries"]):
                    try: out["retries"] = int(RE_INT.search(str(v)).group())
                    except Exception: pass
                if lk in (x.lower() for x in YAML_KEYS["matrix"]):
                    if isinstance(v, dict):
                        for ax, _vals in v.items():
                            out["matrix_axes"].add(str(ax))
                if lk in (x.lower() for x in YAML_KEYS["runner_os"]):
                    out["runner_os"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["jdk"]):
                    out["jdk"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["invocation"]):
                    _invocation_snippets.append(str(v))
                if lk in (x.lower() for x in YAML_KEYS["thirdparty"]):
                    out["thirdparty_refs"].add(lk)
                if isinstance(v, (dict, list)):
                    scan_obj(v)
        elif isinstance(obj, list):
            for x in obj:
                scan_obj(x)

    for d in docs:
        scan_obj(d)

    # classify invocation using YAML snippets + Gradle context (if provided)
    style, tags = classify_invocation_from_texts(_invocation_snippets, gradle_texts or [])
    out["invocation_style"] = style
    out["invocation_tags"].update(tags)

    # convert sets to sorted lists for JSON
    out["api_levels"]      = sorted(out["api_levels"])
    out["abis"]            = sorted(out["abis"])
    out["system_images"]   = sorted(out["system_images"])
    out["device_profiles"] = sorted(out["device_profiles"])
    out["matrix_axes"]     = sorted(out["matrix_axes"])
    out["invocation_tags"] = sorted(out["invocation_tags"])
    out["thirdparty_refs"] = sorted(out["thirdparty_refs"])
    return out

# -----------------------------
# Gradle helpers (tags + richer detection)
# -----------------------------
AGP_PLUGIN_DSL_RX = re.compile(
    r"""id\s*\(?\s*      # id(
        [\"']com\.android\.(?:application|library|test|dynamic-feature)[\"']\s*\)?   # id("com.android.xyz")
        \s*version\s*
        [\"']([^\"']+)[\"']                      # version "X.Y.Z"
    """,
    re.I | re.X,
)

ORCHESTRATOR_COORD_RX = re.compile(r"androidx\.test:orchestrator(?::[^\s'\"\)]+)?", re.I)
ORCHESTRATOR_EXEC_RX  = re.compile(r"testOptions\s*\{[^}]*execution\s*['\"]ANDROIDX_TEST_ORCHESTRATOR['\"]", re.I | re.S)
ORCHESTRATOR_FLAG_RX  = re.compile(r"\buseOrchestrator\s*(?:=|\s)\s*true\b", re.I)
ORCHESTRATOR_PROP_RX  = re.compile(r"\bandroid(?:\.testInstrumentationRunnerArguments)?\.use(?:Test)?Orchestrator\s*=\s*true", re.I)

def extract_gradle_invocation_tags(text: str) -> Set[str]:
    return _collect_tags_from_text(text or "", TAG_PATTERNS_GRADLE)

def detect_orchestrator_from_gradle(text: str) -> bool:
    t = text or ""
    return bool(
        ORCHESTRATOR_COORD_RX.search(t) or
        ORCHESTRATOR_EXEC_RX.search(t)  or
        ORCHESTRATOR_FLAG_RX.search(t)  or
        ORCHESTRATOR_PROP_RX.search(t)
    )

# -----------------------------
# Single-file extractor (YAML + Gradle)
# -----------------------------
def extract_from_text(path: str, text: str, gradle_context_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    data: Dict[str, Any] = {}

    # YAML configs (with gradle context for richer classification)
    if path.lower().endswith((".yml", ".yaml")) and yaml is not None:
        data = extract_from_yaml_text(text, gradle_context_texts or [])

    # Gradle heuristics
    if path.endswith(("build.gradle", "build.gradle.kts",
                      "gradle.properties", "gradle/wrapper/gradle-wrapper.properties",
                      "settings.gradle", "settings.gradle.kts")):
        # AGP via dependency coordinates (classpath or anywhere)
        dep_agp = re.findall(r"com\.android\.tools\.build:gradle:([0-9][^'\"\s\)]+)", text)
        # AGP via plugins { id("com.android.application") version "X" }
        dsl_agp = AGP_PLUGIN_DSL_RX.findall(text)
        agp_all = sorted(set(dep_agp + dsl_agp))
        if agp_all:
            data["agp_versions"] = agp_all

        # apiLevel = N (managed devices DSL or custom config)
        for m in re.finditer(r"\bapiLevel\s*=\s*(\d+)", text, re.IGNORECASE):
            lvl = int(m.group(1))
            data.setdefault("api_levels", [])
            if lvl not in data["api_levels"]:
                data["api_levels"].append(lvl)

        # Orchestrator signals
        if "ANDROIDX_TEST_ORCHESTRATOR" in text or detect_orchestrator_from_gradle(text):
            data["orchestrator"] = True

        # Recognize GMD via Gradle DSL (prefer gmd if present)
        if GMD_GRADLE_RX.search(text):
            current = data.get("invocation_style")
            if current in (None, "", "unknown", "diy", "gradle_connected"):
                data["invocation_style"] = "gmd"

        # Invocation tags from Gradle DSL / properties
        gtags = extract_gradle_invocation_tags(text)
        if gtags:
            data.setdefault("invocation_tags", [])
            merged = sorted(set(data["invocation_tags"]) | gtags)
            data["invocation_tags"] = merged

        # Gradle wrapper version (optional but handy)
        if path.endswith("gradle/wrapper/gradle-wrapper.properties"):
            m = re.search(r"distributionUrl=.*?/gradle-([0-9][\w\.\-]+)-", text)
            if m:
                data["gradle_wrapper_version_raw"] = m.group(1)

    return data

# -----------------------------
# IO helpers
# -----------------------------
def write_jsonl(path: Path, rows: List[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# %%
# -----------------------------
# Main mining loop
# -----------------------------
repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
repos.sort(key=lambda p: p.name.lower())
print(f"Found {len(repos)} repos in {CLONE_ROOT}")

ok_count = 0
skip_count = 0
err_count = 0

for repo in repos:
    try:
        rel_commits = list_relevant_commits(repo)
        if MAX_COMMITS_PER_REPO > 0:
            rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]

        out_rows: List[dict] = []
        for sha, ts, changed_paths in rel_commits:
            # keep only the relevant paths from that commit
            rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
            if not rel_paths:
                continue

            # Load all relevant file texts for this commit first (to provide Gradle context to YAML)
            path_texts: Dict[str, Optional[str]] = {}
            for pth in rel_paths:
                path_texts[pth] = git_show(repo, sha, pth)

            # Gather Gradle texts for this commit
            gradle_texts = [txt for pth, txt in path_texts.items() if txt is not None and is_gradle_file(pth)]

            subj = git_subject(repo, sha)
            for pth in rel_paths:
                txt = path_texts.get(pth)
                if txt is None:
                    continue
                feats = extract_from_text(pth, txt, gradle_context_texts=gradle_texts)
                out_rows.append({
                    "repo": repo.name,
                    "sha": sha,
                    "timestamp": ts,
                    "subject": subj,
                    "path": pth,
                    "features": feats,
                })

        if not out_rows:
            print(f"[skip] {repo.name}: no relevant snapshots")
            skip_count += 1
            continue

        dst = SNAPSHOT_DIR / f"{repo.name}.jsonl"
        write_jsonl(dst, out_rows)
        print(f"[ok] {repo.name}: {len(out_rows)} snapshots -> {dst}")
        ok_count += 1

    except Exception as e:
        # Never crash the whole batch; log and continue
        print(f"[err] {repo.name}: {e}")
        err_count += 1

print(f"\nDone. ok={ok_count}, skip={skip_count}, err={err_count}, out_dir={SNAPSHOT_DIR}")


Found 282 repos in C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
[ok] 4eRTuk__audioview: 63 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 171 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 158 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 48 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 131 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\abdelaziz-mahdy__pytorch_lite.jsonl
[ok] ably__ably-flutter: 316 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\ably__ably-flutter.jsonl
[ok] AChep__AcDisplay: 121 snapshots -> C:\Android Mobile App

In [14]:
# RQ2 — Step 2 (Label+ V3.1): Field-level deltas + commit/delta secondary labels + Driver
# - Commit-side secondary labels: HARDCODED regex (no CSV).
# - Delta-side secondary labels: deterministic, from field-level diffs.
# - Path-driven signal: intent_path ∈ {ci_env_workflow, version_change, coverage, None}
#     * Deterministic, no confidence scoring
#     * Fill-blank ONLY (never overrides commit/delta)
#     * intent_path_reason recorded for audit
# - Emits driver_source ∈ {"commit_delta","path_fill","none"} and intent_path_applied ∈ {0,1}
# - UTC timestamps: epoch + ISO-8601 Z kept for compatibility
# - Aligns with Step 1 "invocation_style" + "invocation_tags"; primary "orchestrator_change"
# - Preserves invocation_tags deltas for later analysis.

from __future__ import annotations
import json, re
import datetime as _dt
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"          # input from Step 1
CCE_ENRICHED_DIR  = WORK_ROOT / "cce_enriched_V3.1"  # output per-repo JSONL (V3.1)
CCE_ENRICHED_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
VERSION_RE = re.compile(r"\d+(?:\.\d+)*")

# Primary label mapping (what changed)
FIELD_TO_PRIMARY = {
    "api_levels":        "api_bump",
    "agp_versions":      "agp_bump",
    "jdk":               "jdk_bump",
    "runner_os":         "runner_os_change",
    "matrix_axes":       "matrix_change",
    "orchestrator":      "orchestrator_change",
    "timeouts":          "timeout_tuning",
    "retries":           "retry_tuning",
    "device_profiles":   "device_profile_change",
    "abis":              "abi_change",
    "system_images":     "system_image_change",
    "invocation_style":  "invocation_change",
    "invocation_tags":   "invocation_change",
    "thirdparty_refs":   "external_service_change",
    "wait_for_boot":     "wait_strategy_change",
}

def read_snapshots(folder: Path) -> Dict[str, List[dict]]:
    by_repo: Dict[str, List[dict]] = {}
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                repo = d.get("repo")
                if not repo:
                    continue
                by_repo.setdefault(repo, []).append(d)
    for repo, rows in by_repo.items():
        rows.sort(key=lambda r: (r.get("path",""), int(r.get("timestamp", 0)), r.get("sha","")))
    return by_repo

def as_set_str(xs) -> set:
    if xs is None:
        return set()
    if isinstance(xs, (list, set, tuple)):
        return set(str(x) for x in xs)
    return {str(xs)}

def as_set_int(xs) -> set:
    if xs is None:
        return set()
    out = set()
    if isinstance(xs, (list, set, tuple)):
        for x in xs:
            try:
                out.add(int(x))
            except Exception:
                pass
    else:
        try:
            out.add(int(xs))
        except Exception:
            pass
    return out

def parse_version_tuple(s: str) -> Tuple[int, ...]:
    if not s:
        return tuple()
    m = VERSION_RE.search(str(s))
    if not m:
        return tuple()
    parts = m.group(0).split(".")
    out: List[int] = []
    for p in parts:
        try:
            out.append(int(p))
        except Exception:
            out.append(0)
    return tuple(out)

def max_version_tuple(strings: List[str]) -> Tuple[int, ...]:
    best: Tuple[int, ...] = tuple()
    for s in strings or []:
        vt = parse_version_tuple(str(s))
        if vt > best:
            best = vt
    return best

def stringify(x: Any) -> str:
    if isinstance(x, (dict, list, set, tuple)):
        try:
            return json.dumps(x, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(x)
    return "" if x is None else str(x)

def _safe_json_loads(s: str):
    try:
        return json.loads(s) if s else None
    except Exception:
        return None

def diff_features(old: Dict[str, Any], new: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Returns per-field deltas:
      [{field, old_value, new_value, change_type, magnitude?, added_items?, removed_items?}, ...]
    Values are stringified JSON; callers can _safe_json_loads().
    """
    old = old or {}
    new = new or {}
    out: List[Dict[str, Any]] = []

    def handle_set(field: str, to_set_fn):
        a = to_set_fn(old.get(field))
        b = to_set_fn(new.get(field))
        if a == b:
            return
        added   = sorted(b - a)
        removed = sorted(a - b)
        row = {
            "field": field,
            "old_value": stringify(sorted(a)),
            "new_value": stringify(sorted(b)),
            "change_type": "modified",
        }
        if added:   row["added_items"] = stringify(added)
        if removed: row["removed_items"] = stringify(removed)
        if field == "api_levels" and a and b:
            try:
                row["magnitude"] = max(b) - max(a)
            except Exception:
                pass
        out.append(row)

    # Set-like fields
    handle_set("api_levels", as_set_int)
    handle_set("abis", as_set_str)
    handle_set("system_images", as_set_str)
    handle_set("device_profiles", as_set_str)
    handle_set("matrix_axes", as_set_str)
    handle_set("invocation_tags", as_set_str)
    handle_set("thirdparty_refs", as_set_str)

    # AGP versions (ordered strings → compare as sets + track direction)
    a_agp = sorted(as_set_str(old.get("agp_versions")))
    b_agp = sorted(as_set_str(new.get("agp_versions")))
    if a_agp != b_agp:
        row = {
            "field": "agp_versions",
            "old_value": stringify(a_agp),
            "new_value": stringify(b_agp),
            "change_type": "modified",
        }
        old_max = max_version_tuple(list(a_agp))
        new_max = max_version_tuple(list(b_agp))
        if old_max or new_max:
            row["magnitude"] = 1 if new_max > old_max else (-1 if new_max < old_max else 0)
        out.append(row)

    # Scalar fields (including invocation_style)
    for field in ("orchestrator","wait_for_boot","retries","runner_os","jdk","invocation_style"):
        a = old.get(field, None)
        b = new.get(field, None)
        if a != b:
            out.append({
                "field": field,
                "old_value": stringify(a),
                "new_value": stringify(b),
                "change_type": (
                    "modified" if (a is not None and b is not None)
                    else ("added" if a is None else "removed")
                ),
            })

    # Dict-ish fields
    for field in ("timeouts",):
        a = old.get(field, None)
        b = new.get(field, None)
        if stringify(a) != stringify(b):
            out.append({
                "field": field,
                "old_value": stringify(a),
                "new_value": stringify(b),
                "change_type": (
                    "modified" if (a is not None and b is not None)
                    else ("added" if a is None else "removed")
                ),
            })

    return out

# -----------------------------
# Commit intent patterns (HARDCODED) — no revert suppression
# -----------------------------
@dataclass
class IntentPattern:
    label: str
    regex: re.Pattern

def _build_hardcoded_intent_patterns() -> List[IntentPattern]:
    patterns: List[Tuple[str, str]] = [
        # Reverts (now just another label; no special handling)
        ("revert", r"(?i)\b(?:revert(?:ed|ing|s)?|roll(?:ed|ing)?(?:\s*[-_ ]\s*)?back|rollback(?:s)?|back(?:ed|ing)?(?:\s*[-_ ]\s*)?out|backout(?:s)?|undo(?:es|ing)?|undid|undone)\b"),

        # Stability / CI-speed
        ("deflake",   r"(?i)\b(?:de[-_ ]?flake\w*|flak(?:e|y|iness)|regression(?:s)?|regress(?:ed|es|ing))\b"),
        ("fail_fix",  r"(?i)\b(?:fix\w*|hotfix\w*|crash|error|timeout|broken|fail(?:ing)?)\b"),
        ("speed_up",  r"(?i)(?:\b(?:speed(?:\s*-\s*)?up|speedup|faster)\b.{0,20}\b(?:ci|build)s?\b|\b(?:ci|build)s?\b.{0,20}\b(?:speed(?:\s*-\s*)?up|speedup|faster)\b|\b(?:reduce|shorten)\b.{0,12}\b(?:ci|build)\b\s*(?:time|times|duration|durations)\b)"),

        # CI platform / config
        ("ci_migration",
         r"(?i)(?:\b(migrat(?:e|ion)|switch|move|replace)\b.*\b(github actions|\bgha\b|gitlab|jenkins|circleci|azure pipelines|bitrise)\b|\b(github actions|\bgha\b|gitlab|jenkins|circleci|azure pipelines|bitrise)\b.*\b(migrat(?:e|ion)|switch|move|replace)\b)"),
        ("ci_config_change",
         r"(?i)(?!.*\b(migrat(?:e|ion)|switch|move|replace)\b.*\b(github actions|\bgha\b|gitlab|jenkins|circleci|azure pipelines|bitrise)\b)"
         r"(?!.*\b(github actions|\bgha\b|gitlab|jenkins|circleci|azure pipelines|bitrise)\b.*\b(migrat(?:e|ion)|switch|move|replace)\b)"
         r"(?:\b(?:workflow|workflows|github(?:[-\s])?actions?|gh\s*actions?|\bgha\b|gitlab|pipeline|pipelines|runner|trigger|action|actions|job|jobs|matrix)\b|\.ci/|\.github/workflows/|\bci\.ya?ml\b|setup[-\s]?java|uses:\s*actions/setup-java|\b(?:provide|configure|config|set(?:\s*up)?|require(?:ment)?|use|install|specify)\b.{0,20}\bci\b)"),

        # Housekeeping / meta
        ("cleanup_refactor",     r"(?i)\b(clean ?up|refactor|rename|format|lint|simplif\w+)\b"),
        ("release_tagging",      r"(?i)\b(release|tag(?:ging)?|rc|beta|alpha|nightly|prepare release)\b"),
        ("removal_deprecation",  r"(?i)\b(remove|delete|drop|deprecat\w+)\b"),
        ("docs_changelog",       r"(?i)\b(docs?|readme|changelog)\b"),

        # Version cues — shorter two-way pattern (NO revert exclusion)
        ("version_change",
         r"(?i)\b(?:(?:\bagp\b|\bgradle(?:\s*wrapper)?\b|\bjdk\b|\bjava\b|\btoolchain\b).{0,24}\b(?:\d+(?:\.\d+)+|1[1-9]|[89])\b|\b(?:\d+(?:\.\d+)+|1[1-9]|[89])\b.{0,24}(?:\bagp\b|\bgradle(?:\s*wrapper)?\b|\bjdk\b|\bjava\b|\btoolchain\b)|\bversion\s+\d+(?:\.\d+)*(?:\s*(?:--?>|=>|->|>)\s*\d+(?:\.\d+)*)?\b)"),

        # Update/upgrade — flexible ends, includes downgrade; NO revert exclusion
        ("update_upgrade",
         r"(?i)\b(?:bump|pin|unpin\w*|upgrad\w*|updat\w*|downgrad\w*|adopt\w*|refresh\w*)\b"),
    ]
    return [IntentPattern(label=lbl, regex=re.compile(rx, re.MULTILINE)) for lbl, rx in patterns]

INTENT_PATTERNS: List[IntentPattern] = _build_hardcoded_intent_patterns()

# -----------------------------
# Intent taxonomy & driver mapping
# -----------------------------
AUX_LABELS = {
    "revert", "cleanup_refactor", "removal_deprecation",
    "docs_changelog", "chore", "release_tagging"
}  # recorded, never used to choose driver

LABEL_TO_CATEGORY = {
    # commit-side → driver categories
    "deflake": "stability_tuning",
    "fail_fix": "stability_tuning",
    "speed_up": "stability_tuning",
    "ci_migration": "ci_env_workflow",
    "ci_config_change": "ci_env_workflow",
    "update_upgrade": "version_change",
    "version_change": "version_change",

    # delta-side → driver categories
    "expand_coverage": "coverage",
    "reduce_coverage": "coverage",
    "coverage_dimensions_change": "coverage",
    "timeout_tuning": "stability_tuning",
    "retry_tuning": "stability_tuning",
    "orchestrator_change": "stability_tuning",
    "flake_mitigation": "stability_tuning",
    "speed_up_ci": "stability_tuning",
    "infra_tuning": "ci_env_workflow",
    "external_service_change": "ci_env_workflow",
    "adopt_gmd": "version_change",
    "drop_gmd": "version_change",
    "adopt_diy": "version_change",
}

DRIVER_PRIORITY = ["stability_tuning", "coverage", "ci_env_workflow", "version_change"]

# -----------------------------
# Commit-side secondary labels
# -----------------------------
def derive_commit_labels(subject: str) -> Tuple[List[str], List[str]]:
    """
    Returns (commit_labels, aux_tags).
    All matched labels are kept (including 'revert'). Aux tags are returned separately
    so the driver resolver can ignore them.
    """
    s = subject or ""
    labels: set[str] = set()
    for ip in INTENT_PATTERNS:
        try:
            if ip.regex.search(s):
                labels.add(ip.label)
        except Exception:
            continue
    aux = sorted(lbl for lbl in labels if lbl in AUX_LABELS)
    non_aux = sorted(lbl for lbl in labels if lbl not in AUX_LABELS)
    return non_aux + aux, aux

# -----------------------------
# Delta-side secondary labels
# -----------------------------
_DURATION_RX = re.compile(r"(?i)^\s*(\d+(?:\.\d+)?)\s*(ms|s|m|h)?\s*$")
_UNIT_TO_SEC = {"ms": 0.001, "s": 1, "m": 60, "h": 3600}

def _to_seconds(x) -> Optional[float]:
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    m = _DURATION_RX.match(str(x))
    if not m:
        return None
    val = float(m.group(1))
    unit = (m.group(2) or "s").lower()
    return val * _UNIT_TO_SEC.get(unit, 1)

def _sum_timeout_seconds(obj) -> Optional[float]:
    """
    Accept dict-like {'setup': '30s', 'test':'90s'} or list of pairs.
    Returns total seconds if at least one value parsed; else None.
    """
    if obj is None:
        return None
    total = 0.0
    seen = 0
    if isinstance(obj, dict):
        it = obj.values()
    elif isinstance(obj, list):
        it = [v for _, v in obj if isinstance(_, (str,int))]
    else:
        return None
    for v in it:
        secs = _to_seconds(v)
        if secs is not None:
            total += secs
            seen += 1
    return total if seen else None

def derive_delta_labels(deltas: List[Dict[str, Any]]) -> List[str]:
    labs: set[str] = set()
    for d in deltas:
        field = d.get("field")
        old_v = d.get("old_value")
        new_v = d.get("new_value")
        added = _safe_json_loads(d.get("added_items") or "") or []
        removed = _safe_json_loads(d.get("removed_items") or "") or []

        # Coverage sets
        if field in {"api_levels", "abis", "device_profiles", "system_images"}:
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Matrix axes → coverage dimensions + expand/reduce
        if field == "matrix_axes":
            labs.add("coverage_dimensions_change")
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Timeouts
        if field == "timeouts":
            labs.add("timeout_tuning")
            old_obj = _safe_json_loads(old_v)
            new_obj = _safe_json_loads(new_v)
            old_s = _sum_timeout_seconds(old_obj)
            new_s = _sum_timeout_seconds(new_obj)
            if old_s is not None and new_s is not None:
                if new_s > old_s:
                    labs.add("flake_mitigation")
                elif new_s < old_s:
                    labs.add("speed_up_ci")

        # Retries
        if field == "retries":
            labs.add("retry_tuning")
            try:
                a = int(_safe_json_loads(old_v) if old_v else 0)
            except Exception:
                a = None
            try:
                b = int(_safe_json_loads(new_v) if new_v else 0)
            except Exception:
                b = None
            if a is not None and b is not None:
                if b > a:
                    labs.add("flake_mitigation")
                elif b < a:
                    labs.add("speed_up_ci")

        # Orchestrator
        if field == "orchestrator":
            labs.add("orchestrator_change")

        # Runner platform
        if field == "runner_os":
            labs.add("infra_tuning")

        # Invocation strategy flips
        if field == "invocation_style":
            old_s = (old_v or "").strip().lower()
            new_s = (new_v or "").strip().lower()
            if new_s == "gmd" and old_s != "gmd":
                labs.add("adopt_gmd")
            if old_s == "gmd" and new_s != "gmd":
                labs.add("drop_gmd")
            if new_s == "diy" and old_s != "diy":
                labs.add("adopt_diy")

        # External device-cloud integrations
        if field == "thirdparty_refs":
            labs.add("external_service_change")

        # Note: we do not map invocation_tags adds/removes to delta labels here.
    return sorted(labs)

# -----------------------------
# Driver resolver
# -----------------------------
def resolve_driver(secondary_labels_union: List[str], aux_tags: List[str]) -> Optional[str]:
    # Remove aux labels from consideration
    effective = [lbl for lbl in secondary_labels_union if lbl not in AUX_LABELS]
    # Map labels to categories
    cats: set[str] = set()
    for lbl in effective:
        cat = LABEL_TO_CATEGORY.get(lbl)
        if cat:
            cats.add(cat)
    # Choose by fixed priority
    for cat in DRIVER_PRIORITY:
        if cat in cats:
            return cat
    return None  # no non-aux secondary labels → driver remains null

# -----------------------------
# Path-driven intent (deterministic, no confidence)
# -----------------------------
_CI_PATH_RX = re.compile(
    r"(?i)(?:^|/)(?:\.github/workflows?|\.circleci|\.gitlab-ci\.yml|azure-pipelines\.yml|bitrise\.yml|jenkinsfile|\.jenkins|ci|\.ci)(?:/|$)"
)
_GRADLE_PATH_RX = re.compile(
    r"(?i)(?:^|/)(?:build|settings)\.gradle(?:\.kts)?$|(?:^|/)gradle\.properties$|(?:^|/)gradle/wrapper/gradle-wrapper\.properties$"
)
_SCRIPT_PATH_RX = re.compile(
    r"(?i)(?:^|/)(?:scripts?|tools?)(?:/|$)|\.(sh|py|bat|ps1)$"
)

def _is_ci_path(path: str) -> bool:
    return bool(_CI_PATH_RX.search(path or ""))

def _is_gradle_path(path: str) -> bool:
    return bool(_GRADLE_PATH_RX.search(path or ""))

def _is_script_path(path: str) -> bool:
    return bool(_SCRIPT_PATH_RX.search(path or ""))

_COVERAGE_FIELDS = {"api_levels", "abis", "device_profiles", "system_images", "matrix_axes"}

def infer_intent_path(path: str, field: str) -> tuple[str, str]:
    """
    returns (intent_path, intent_path_reason) or ("","") if none.
    Rules:
      - CI paths + coverage field -> coverage
      - CI paths -> ci_env_workflow
      - Gradle paths + field in {agp_versions, jdk} -> version_change
      - Script paths -> ci_env_workflow
    """
    p = path or ""
    f = (field or "").lower()

    if _is_ci_path(p) and f in _COVERAGE_FIELDS:
        return ("coverage", "ci path + coverage field")

    if _is_ci_path(p):
        return ("ci_env_workflow", ".ci/.github/workflows/etc")

    if _is_gradle_path(p):
        if f in {"agp_versions", "jdk"}:
            return ("version_change", "gradle/build tool path")
        return ("", "")

    if _is_script_path(p):
        return ("ci_env_workflow", "script/tool path")

    return ("", "")

# -----------------------------
# UTC timestamp helpers
# -----------------------------
def ensure_utc_epoch(ts_any) -> int:
    try:
        ts = int(float(ts_any))
    except Exception:
        ts = 0
    return ts

def epoch_to_iso_utc(ts: int) -> str:
    return _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc).isoformat().replace("+00:00", "Z")

# -----------------------------
# Main: build enriched episodes
# -----------------------------
if __name__ == "__main__":
    print(f"[info] snapshots dir: {SNAPSHOT_DIR}")
    print(f"[info] output dir   : {CCE_ENRICHED_DIR}")
    print("[commit-intents]", [p.label for p in INTENT_PATTERNS])

    by_repo = read_snapshots(SNAPSHOT_DIR)
    print(f"[info] Loaded snapshots for {len(by_repo)} repos")

    for repo, rows in by_repo.items():
        out_rows: List[dict] = []
        by_path: Dict[str, List[dict]] = {}
        for r in rows:
            by_path.setdefault(r.get("path",""), []).append(r)

        # Track repeats per (path, field) within the repo
        repeat_counter: Dict[Tuple[str,str], int] = {}

        for path, snaps in by_path.items():
            prev: Optional[dict] = None
            for cur in snaps:
                if prev is None:
                    prev = cur
                    continue

                old_feats = prev.get("features", {}) or {}
                new_feats = cur.get("features", {}) or {}
                deltas = diff_features(old_feats, new_feats)

                if deltas:
                    # Episode-level (prev -> cur) labels
                    commit_labels_all, aux_tags = derive_commit_labels(cur.get("subject",""))
                    delta_labels_all = derive_delta_labels(deltas)
                    secondary_union = sorted(set(commit_labels_all) | set(delta_labels_all))
                    episode_driver = resolve_driver(secondary_union, aux_tags)

                    # Timestamp: force UTC
                    ts_epoch_utc = ensure_utc_epoch(cur.get("timestamp", 0))
                    ts_iso_utc   = epoch_to_iso_utc(ts_epoch_utc)

                    for d in deltas:
                        field = d["field"]
                        key = (path, field)
                        repeat_counter[key] = repeat_counter.get(key, 0) + 1
                        repeat_index = repeat_counter[key]
                        repeat_label = "first" if repeat_index == 1 else "repeat"

                        primary_label = FIELD_TO_PRIMARY.get(field, "other_change")

                        # Path-driven intent (advisory only)
                        intent_path, intent_path_reason = infer_intent_path(path, field)

                        # Fill-blank ONLY (do not override commit/delta)
                        if episode_driver:
                            driver_row = episode_driver
                            driver_source = "commit_delta"
                            intent_path_applied = 0
                        elif intent_path:
                            driver_row = intent_path
                            driver_source = "path_fill"
                            intent_path_applied = 1
                        else:
                            driver_row = None
                            driver_source = "none"
                            intent_path_applied = 0

                        out_rows.append({
                            "repo": repo,
                            "sha": cur.get("sha"),
                            "prev_sha": prev.get("sha"),

                            # UTC timestamps (compat)
                            "timestamp_epoch_utc": ts_epoch_utc,
                            "timestamp_utc": ts_iso_utc,
                            "timestamp": ts_epoch_utc,  # legacy

                            "path": path,
                            "subject": cur.get("subject",""),

                            # Field-level delta
                            "field": field,
                            "old_value": d.get("old_value",""),
                            "new_value": d.get("new_value",""),
                            "change_type": d.get("change_type","modified"),
                            "magnitude": d.get("magnitude", None),
                            "added_items": d.get("added_items",""),
                            "removed_items": d.get("removed_items",""),

                            "repeat_index": repeat_index,
                            "repeat_label": repeat_label,

                            # Labels
                            "primary_label": primary_label,
                            "secondary_labels_commit": sorted(commit_labels_all),
                            "secondary_labels_delta": sorted(delta_labels_all),
                            "secondary_labels": secondary_union,
                            "aux_tags": aux_tags,

                            # Path-driven intent (advisory only)
                            "intent_path": intent_path or None,
                            "intent_path_reason": intent_path_reason if intent_path else None,
                            "intent_path_applied": intent_path_applied,   # 1/0

                            # Driver
                            "driver": driver_row,
                            "driver_source": driver_source,
                        })
                prev = cur

        if not out_rows:
            print(f"[skip] {repo}: no field-level deltas found")
            continue

        dst = CCE_ENRICHED_DIR / f"{repo}.jsonl"
        with dst.open("w", encoding="utf-8") as f:
            for r in out_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[ok] {repo}: {len(out_rows)} enriched rows -> {dst}")

    print("Done.")


[info] snapshots dir: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots
[info] output dir   : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.1
[commit-intents] ['revert', 'deflake', 'fail_fix', 'speed_up', 'ci_migration', 'ci_config_change', 'cleanup_refactor', 'release_tagging', 'removal_deprecation', 'docs_changelog', 'version_change', 'update_upgrade']
[info] Loaded snapshots for 282 repos
[ok] 4eRTuk__audioview: 8 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.1\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 16 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.1\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 18 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.1\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 3 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\

In [15]:
# %% [markdown]
# RQ2 — Step 3 (Combine+ V3.1): Snapshots + Enriched Episodes (+ change_op fields)
# Combines per-repo JSONL into CSV (and Parquet if pandas/pyarrow available).
# - Snapshots: keep `features_json` as a string (drop raw `features`)
# - Episodes (from Step 2, V3.1):
#     * Keep only ONE format for secondary labels: sorted, deduped, comma-separated strings (no [] or quotes)
#       - Columns kept: `secondary_labels`, `secondary_labels_commit`, `secondary_labels_delta`, `aux_tags`
#       - Also keep counts/flags: `*_count`, `has_*`
#     * Keep only `timestamp_utc` (drop `timestamp` and `timestamp_epoch_utc` from episodes)
#     * `driver` (as produced in Step 2) + `has_driver`
#     * NEW: pass through `intent_path`, `intent_path_reason`
#     * NEW: annotate when driver was filled by path:
#           - `driver_source` in {"commit_delta","path_fill","none"}
#           - `intent_path_applied` in {0,1}
#     * `change_op` in {"add","remove","value_edit","no_change"}
#     * `is_major_emulator_change` (1 if add/remove OR coverage set grew/shrank; else 0)
# - Parquet: enforce predictable dtypes

from __future__ import annotations
import json, csv, os, re, ast, math
from pathlib import Path
from typing import List, Any, Optional, Tuple

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"             # Step 1 output

# Prefer latest Step-2 output; fallback for compatibility
CCE_ENRICHED_DIRS = [
    WORK_ROOT / "cce_enriched_V3.1",   # NEW (Label+ V3.1 with intent_path)
    WORK_ROOT / "cce_enriched_V3.0",   # fallback
]
CCE_ENRICHED_DIR = next((p for p in CCE_ENRICHED_DIRS if p.exists()), CCE_ENRICHED_DIRS[0])

COMBINE_DIR       = WORK_ROOT / "combined_V3.0"
COMBINE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def read_all_jsonl(folder: Path) -> List[dict]:
    out: List[dict] = []
    if not folder.exists():
        return out
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    d["_source_file"] = p.name
                    out.append(d)
    return out

def to_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return "" if x is None else str(x)

def to_list(x: Any) -> list:
    """Tolerant listifier for list/str/json-str/None."""
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        # try JSON list
        if s.startswith("[") and s.endswith("]"):
            try:
                v = json.loads(s)
                return v if isinstance(v, list) else [s]
            except Exception:
                pass
        # try Python literal list/tuple
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple, set)):
                return list(v)
        except Exception:
            pass
        # try semicolon-separated
        if ";" in s:
            parts = [t.strip() for t in s.split(";") if t.strip()]
            return parts
        # otherwise treat as one label string
        return [s] if s else []
    return [x]

def normalize_label_string(val: Any) -> tuple[str, int, bool]:
    """
    Return (comma_joined, count, has_any).
    - Deduplicates and sorts labels.
    - Produces a clean comma-separated string: a,b,c (no brackets or quotes).
    """
    items = [str(t).strip() for t in to_list(val) if str(t).strip()]
    items = sorted(set(items))
    s = ",".join(items)
    return s, len(items), bool(items)

# -----------------------------
# Change classification helpers
# -----------------------------
CANDIDATE_COL_PAIRS = [
    ("old_value", "new_value"),
    ("value_old", "value_new"),
    ("prev_value", "curr_value"),
    ("previous_value", "current_value"),
    ("before", "after"),
    ("value_before", "value_after"),
    ("lhs", "rhs"),
    ("from_value", "to_value"),
    ("from", "to"),
    ("baseline_value", "value"),
]

def _is_na_like(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        try:
            return math.isnan(x)
        except Exception:
            return False
    if isinstance(x, str):
        s = x.strip().lower()
        return s in {"", "null", "none", "nan", "na"}
    return False

def to_none_if_empty(x: Any):
    return None if _is_na_like(x) else x

def normalize_scalar(s: str):
    if not isinstance(s, str):
        return s
    s2 = s.strip()
    if s2 == "":
        return ""
    try:
        return float(s2)  # numeric normalization
    except Exception:
        pass
    # collapse intra-string whitespace
    return re.sub(r"\s+", " ", s2)

def try_literal_or_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return ast.literal_eval(s)  # e.g., "['arm64','x86_64']"
    except Exception:
        return s

def make_json_safe(obj):
    if isinstance(obj, float):
        return obj
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    return obj

def canonicalize(v):
    v = to_none_if_empty(v)
    if v is None:
        return None
    v_parsed = try_literal_or_json(v) if isinstance(v, str) else v
    if isinstance(v_parsed, dict):
        return json.dumps(make_json_safe(v_parsed), sort_keys=True, ensure_ascii=False)
    if isinstance(v_parsed, (list, tuple, set)):
        seq = list(v_parsed)
        normalized = [normalize_scalar(str(x)) if not isinstance(x, (dict, list, tuple, set)) else x for x in seq]
        try:
            return json.dumps(sorted(make_json_safe(normalized)), ensure_ascii=False)
        except Exception:
            return json.dumps(make_json_safe(normalized), ensure_ascii=False)
    if isinstance(v_parsed, str):
        return normalize_scalar(v_parsed)
    return v_parsed

def classify_change(old, new):
    old_c, new_c = canonicalize(old), canonicalize(new)
    if old_c is None and new_c is None:
        return "no_change"
    if old_c is None and new_c is not None:
        return "add"
    if old_c is not None and new_c is None:
        return "remove"
    if old_c != new_c:
        return "value_edit"
    return "no_change"

def find_value_columns_from_records(records: List[dict]) -> Tuple[Optional[str], Optional[str]]:
    """
    Auto-detect the old/new value columns from the union of keys in `records`.
    Returns (left_col_name, right_col_name) in original casing or (None, None).
    """
    # union of keys (original casing)
    all_keys: List[str] = []
    for r in records:
        for k in r.keys():
            if k not in all_keys:
                all_keys.append(k)

    lower_to_orig = {}
    for k in all_keys:
        kl = k.lower()
        if kl not in lower_to_orig:
            lower_to_orig[kl] = k

    keys_lower = set(lower_to_orig.keys())

    # exact pair match
    for left, right in CANDIDATE_COL_PAIRS:
        if left.lower() in keys_lower and right.lower() in keys_lower:
            return lower_to_orig[left.lower()], lower_to_orig[right.lower()]

    # regex fallback (heuristic)
    left_candidates  = [k for k in all_keys if re.search(r"(old|prev|before|baseline)", k, re.I)]
    right_candidates = [k for k in all_keys if re.search(r"(new|curr|after|current|to\b|value$)", k, re.I)]
    if left_candidates and right_candidates:
        return left_candidates[0], right_candidates[0]

    return None, None

def records_have_columns(records: List[dict], cols: List[str]) -> bool:
    lc = [c.lower() for c in cols]
    for r in records:
        ks = {k.lower() for k in r.keys()}
        if all(c in ks for c in lc):
            return True
    return False

# -----------------------------
# Load inputs
# -----------------------------
snapshots = read_all_jsonl(SNAPSHOT_DIR)
episodes  = read_all_jsonl(CCE_ENRICHED_DIR)
print(f"Loaded {len(snapshots)} snapshots from {SNAPSHOT_DIR}; {len(episodes)} enriched episode rows from {CCE_ENRICHED_DIR}.")

# -----------------------------
# Write CSV (snapshots) — flatten features to JSON string
# -----------------------------
snap_csv = COMBINE_DIR / "snapshots_combined.csv"
if snapshots:
    snaps_flat = []
    for r in snapshots:
        feats = r.get("features", {})
        rr = {**r, "features_json": to_json(feats)}
        rr.pop("features", None)  # drop raw features to avoid CSV/Parquet issues
        # NOTE: We leave snapshot timestamps as-is (Step 1 often only has 'timestamp' epoch).
        snaps_flat.append(rr)

    keys = sorted(set().union(*[set(x.keys()) for x in snaps_flat]))
    with snap_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in snaps_flat:
            w.writerow(r)
    print(f"[ok] {snap_csv}")
else:
    print("[warn] No snapshots found.")

# -----------------------------
# Write CSV (episodes enriched) — compact labels + add change_op fields
# -----------------------------
cce_csv = COMBINE_DIR / "episodes_enriched_combined.csv"
if episodes:
    # Prefer canonical column names from Step-2; fallback to detection
    if records_have_columns(episodes, ["old_value", "new_value"]):
        left_col, right_col = "old_value", "new_value"
        print(f"[detect] value columns: old='{left_col}', new='{right_col}' (canonical)")
    else:
        left_col, right_col = find_value_columns_from_records(episodes)
        if left_col and right_col:
            print(f"[detect] value columns: old='{left_col}', new='{right_col}' (heuristic)")
        else:
            print("[detect] No obvious old/new value columns found; 'change_op' will be 'no_change' by default.")

    episodes_flat = []
    coverage_fields = {"api_levels","abis","system_images","device_profiles","matrix_axes"}

    for r in episodes:
        # Compact secondary label strings (sorted, deduped, comma-separated; no []):
        sec_union_str, sec_union_cnt, sec_union_has = normalize_label_string(r.get("secondary_labels"))
        sec_commit_str, sec_commit_cnt, sec_commit_has = normalize_label_string(r.get("secondary_labels_commit"))
        sec_delta_str,  sec_delta_cnt,  sec_delta_has  = normalize_label_string(r.get("secondary_labels_delta"))
        aux_str,        aux_cnt,        aux_has        = normalize_label_string(r.get("aux_tags"))

        # Primary label
        primary = r.get("primary_label", "unknown")

        # Change op classification
        old_val = r.get(left_col) if left_col else None
        new_val = r.get(right_col) if right_col else None
        change_op = classify_change(old_val, new_val) if (left_col and right_col) else "no_change"

        # Major coverage change: either field was added/removed OR set grew/shrank (value_edit + items)
        added_list   = to_list(r.get("added_items"))
        removed_list = to_list(r.get("removed_items"))
        is_major = int(
            (change_op in ("add", "remove")) or
            (r.get("field") in coverage_fields and (len(added_list) > 0 or len(removed_list) > 0))
        )

        # Driver presence
        driver = r.get("driver", None)
        has_driver = int(bool(driver and str(driver).strip()))

        # Path-driven intent (V3.1) — pass through if present
        intent_path = r.get("intent_path", None)
        intent_path_reason = r.get("intent_path_reason", None)

        # Annotate whether driver was filled by path (blank-fill policy in Step-2)
        # Heuristic: union has no labels AND driver exists AND intent_path exists -> path_fill
        if has_driver and not sec_union_has and intent_path:
            driver_source = "path_fill"
            intent_path_applied = 1
        elif has_driver:
            driver_source = "commit_delta"
            intent_path_applied = 0
        else:
            driver_source = "none"
            intent_path_applied = 0

        rr = {
            **r,

            # Keep only Step-2 driver and timestamp_utc
            "driver": driver,
            "has_driver": has_driver,
            "driver_source": driver_source,           # {"commit_delta","path_fill","none"}
            "intent_path_applied": intent_path_applied,

            # Pass-through of path-driven intent fields (if present)
            "intent_path": intent_path,
            "intent_path_reason": intent_path_reason,

            # Union labels (single compact column + count/flag)
            "secondary_labels": sec_union_str,
            "secondary_label_count": sec_union_cnt,
            "has_secondary_label": int(sec_union_has),

            # Commit-only labels (single compact column + count/flag)
            "secondary_labels_commit": sec_commit_str,
            "secondary_label_commit_count": sec_commit_cnt,
            "has_secondary_label_commit": int(sec_commit_has),

            # Delta-only labels (single compact column + count/flag)
            "secondary_labels_delta": sec_delta_str,
            "secondary_label_delta_count": sec_delta_cnt,
            "has_secondary_label_delta": int(sec_delta_has),

            # Aux tags (single compact column + count/flag)
            "aux_tags": aux_str,
            "aux_tag_count": aux_cnt,
            "has_aux_tag": int(aux_has),

            # Primary (what changed)
            "primary_label": primary,

            # Change classification
            "change_op": change_op,
            "is_major_emulator_change": is_major,
        }

        # Drop redundant episode timestamps; keep only timestamp_utc
        rr.pop("timestamp", None)
        rr.pop("timestamp_epoch_utc", None)

        episodes_flat.append(rr)

    # Final column order: compute from produced rows
    keys = sorted(set().union(*[set(x.keys()) for x in episodes_flat]))
    with cce_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in episodes_flat:
            w.writerow(r)
    print(f"[ok] {cce_csv}")
else:
    print("[warn] No enriched episodes found.")

# -----------------------------
# Optional: Parquet with pandas (safe dtypes)
# -----------------------------
try:
    import pandas as pd

    # Snapshots → Parquet
    if snap_csv.exists():
        df_s = pd.read_csv(snap_csv, dtype=str, keep_default_na=False)

        # Keep snapshot timestamps as-is (Step 1 may only have 'timestamp')
        for col in ("timestamp", "timestamp_epoch_utc"):
            if col in df_s.columns:
                df_s[col] = pd.to_numeric(df_s[col], errors="coerce").astype("Int64")

        df_s.to_parquet(COMBINE_DIR / "snapshots_combined.parquet", index=False)
        print("[ok] Parquet: snapshots_combined.parquet")

    # Enriched episodes → Parquet
    if cce_csv.exists():
        df_e = pd.read_csv(cce_csv, dtype=str, keep_default_na=False)

        # Ensure string dtype for mixed columns
        str_cols = [
            "old_value","new_value","added_items","removed_items","subject","path",
            "driver","driver_source","field","change_type","_source_file","repo","sha","prev_sha",
            "repeat_label","primary_label","timestamp_utc",
            "secondary_labels","secondary_labels_commit","secondary_labels_delta","aux_tags",
            "intent_path","intent_path_reason","change_op",
        ]
        for col in str_cols:
            if col in df_e.columns:
                df_e[col] = df_e[col].astype("string")

        # Numeric-friendly columns (coerce)
        for num_col in (
            "magnitude","repeat_index",
            "secondary_label_count","secondary_label_commit_count","secondary_label_delta_count",
            "aux_tag_count","has_secondary_label","has_secondary_label_commit","has_secondary_label_delta",
            "has_aux_tag","has_driver","is_major_emulator_change","intent_path_applied"
        ):
            if num_col in df_e.columns:
                df_e[num_col] = pd.to_numeric(df_e[num_col], errors="coerce")

        # Nullable integers for indices/flags
        for i_col in ("repeat_index","secondary_label_count","secondary_label_commit_count",
                      "secondary_label_delta_count","aux_tag_count","has_secondary_label",
                      "has_secondary_label_commit","has_secondary_label_delta",
                      "has_aux_tag","has_driver","is_major_emulator_change","intent_path_applied"):
            if i_col in df_e.columns:
                df_e[i_col] = df_e[i_col].astype("Int64")

        df_e.to_parquet(COMBINE_DIR / "episodes_enriched_combined.parquet", index=False)
        print("[ok] Parquet: episodes_enriched_combined.parquet")

except Exception as e:
    print(f"[note] Skipping Parquet (pandas/pyarrow issue?): {e}")


Loaded 186471 snapshots from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots; 12700 enriched episode rows from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.1.
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\snapshots_combined.csv
[detect] value columns: old='old_value', new='new_value' (canonical)
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\episodes_enriched_combined.csv
[ok] Parquet: snapshots_combined.parquet
[ok] Parquet: episodes_enriched_combined.parquet
